In [15]:
import os
import numpy as np
from pymatgen.core import Structure
from pymatgen.core.surface import SlabGenerator
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

# ================= 1. 配置区域 =================
elec_file = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
anode_file = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"
output_dir = "Na_Na3SbS4_Data/Interface_structures-new"

hkl_elec = (0, 0, 1)  
hkl_anode = (0, 0, 1) 

# ZSL 参数
max_area = 400
max_strain = 0.05     
max_angle_tol = 0.1

# ================= 2. 辅助函数 =================
def get_max_strain(interface):
    strain_matrix = None
    if hasattr(interface, "film_strain"):
        strain_matrix = interface.film_strain
    elif hasattr(interface, "metadata"):
        meta = interface.metadata
        if "film_strain" in meta: strain_matrix = meta["film_strain"]
        elif "strain" in meta: strain_matrix = meta["strain"]
    if strain_matrix is not None:
        return np.max(np.abs(strain_matrix))
    return -1.0

# ================= 3. 主逻辑 =================

def run_auto_builder_fixed():
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"📁 创建输出目录: {output_dir}")

    # --- 读取 ---
    if not os.path.exists(elec_file) or not os.path.exists(anode_file):
        print(f"❌ 错误: 找不到输入文件！")
        return
    bulk_elyte = Structure.from_file(elec_file)
    bulk_anode = Structure.from_file(anode_file)

    # --- 切片 ---
    print("🔪 生成基础切片...")
    slab_elyte = SlabGenerator(bulk_elyte, hkl_elec, min_slab_size=9, min_vacuum_size=15, center_slab=True).get_slab()
    slab_anode = SlabGenerator(bulk_anode, hkl_anode, min_slab_size=9, min_vacuum_size=15, center_slab=True).get_slab()

    # --- ZSL ---
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.1,
        max_area=max_area,
        max_length_tol=0.1,
        max_angle_tol=max_angle_tol
    )

    # --- Builder ---
    print("🤖 初始化自动构建器...")
    builder = CoherentInterfaceBuilder(
        substrate_structure=slab_elyte,
        film_structure=slab_anode,
        film_miller=hkl_anode,      
        substrate_miller=hkl_elec,  
        zslgen=zsl                  
    )

    # --- 遍历 ---
    print("🔍 开始搜索并生成界面...")
    saved_count = 0
    available_terms = list(builder.terminations)
    print(f"ℹ️ 发现 {len(available_terms)} 种终端组合。")

    for term in available_terms:
        term_label = f"{term[0]}_{term[1]}" if isinstance(term, tuple) else str(term)
        term_label = term_label.replace(" ", "")

        for interface in builder.get_interfaces(termination=term):
            
            max_s = get_max_strain(interface)
            num_atoms = len(interface)
            
            if max_s != -1 and max_s > max_strain: continue
            if num_atoms > 300 or num_atoms < 40: continue

            saved_count += 1
            
            # ================= [视觉修复核心代码] =================
            # 1. 深度拷贝，防止修改原始对象
            final_struct = interface.copy()
            
            # 2. 居中操作：将所有原子的质心移到晶胞中心 (0.5, 0.5, 0.5)
            # 这样原子就不会挂在盒子的边缘
            try:
                # 计算几何中心 (笛卡尔坐标)
                cart_coords = final_struct.cart_coords
                center_of_mass = np.mean(cart_coords, axis=0)
                
                # 目标中心 (笛卡尔坐标)
                target_center = final_struct.lattice.get_cartesian_coords([0.5, 0.5, 0.5])
                
                # 平移矢量
                translation_vector = target_center - center_of_mass
                
                # 执行平移，并且开启 to_unit_cell=True 
                # 这一步最关键！它会把超出边界 (>1 或 <0) 的原子折叠回盒子内
                final_struct.translate_sites(
                    indices=range(len(final_struct)),
                    vector=translation_vector,
                    frac_coords=False,
                    to_unit_cell=True 
                )
                
                # 3. 按元素排序，美观
                final_struct.sort()
                
            except Exception as e:
                print(f"⚠️ 居中操作失败 (不影响物理正确性): {e}")
                # 如果居中失败，至少做一次折叠
                # final_struct.translate_sites(range(len(final_struct)), [0,0,0], to_unit_cell=True)

            # ================= [结束修复] =================
            
            # 保存修复后的结构
            strain_str = f"{max_s:.3f}" if max_s != -1 else "Unknown"
            file_name = f"Auto_Interface_S{strain_str}_N{num_atoms}_{term_label}_{saved_count}.vasp"
            full_path = os.path.join(output_dir, file_name)
            
            final_struct.to(filename=full_path, fmt="poscar")
            
            print(f"   ✅ [保存] 原子数:{num_atoms} | 应变:{strain_str} | 终端:{term_label}")

            if saved_count >= 5:
                break
        
        if saved_count >= 5:
            print("🛑 已生成 5 个优质结构，停止任务。")
            break

    if saved_count == 0:
        print(f"\n⚠️ 警告: 未找到满足条件的界面。")
    else:
        print(f"\n🎉 任务完成！文件已保存在: {output_dir}")

run_auto_builder_fixed()

📁 创建输出目录: Na_Na3SbS4_Data/Interface_structures-new
🔪 生成基础切片...
🤖 初始化自动构建器...
🔍 开始搜索并生成界面...
ℹ️ 发现 3 种终端组合。
   ✅ [保存] 原子数:52 | 应变:Unknown | 终端:Na_Cmmm_1_S_Cmmm_1
   ✅ [保存] 原子数:52 | 应变:Unknown | 终端:Na_Cmmm_1_S_Cmmm_1
   ✅ [保存] 原子数:56 | 应变:Unknown | 终端:Na_Cmmm_1_S_Cmmm_1
   ✅ [保存] 原子数:56 | 应变:Unknown | 终端:Na_Cmmm_1_S_Cmmm_1
   ✅ [保存] 原子数:80 | 应变:Unknown | 终端:Na_Cmmm_1_S_Cmmm_1
🛑 已生成 5 个优质结构，停止任务。

🎉 任务完成！文件已保存在: Na_Na3SbS4_Data/Interface_structures-new


In [16]:
import os
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.core.surface import SlabGenerator
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

# ================= 1. 配置区域 =================
elec_file = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
anode_file = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"
output_dir = "Na_Na3SbS4_Data/Interface_structures_Vacuum" # 新文件夹名

hkl_elec = (0, 0, 1)  
hkl_anode = (0, 0, 1) 

# ZSL 参数
max_area = 400
max_strain = 0.05     
max_angle_tol = 0.1

# 真空层厚度 (Å) - 这会让它变成单层
VACUUM_SIZE = 20.0 

# ================= 2. 辅助函数 =================
def get_max_strain(interface):
    strain_matrix = None
    if hasattr(interface, "film_strain"):
        strain_matrix = interface.film_strain
    elif hasattr(interface, "metadata"):
        meta = interface.metadata
        if "film_strain" in meta: strain_matrix = meta["film_strain"]
        elif "strain" in meta: strain_matrix = meta["strain"]
    if strain_matrix is not None:
        return np.max(np.abs(strain_matrix))
    return -1.0

# ================= 3. 主逻辑 =================

def run_builder_with_vacuum():
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"📁 创建输出目录: {output_dir}")

    # --- 读取 ---
    if not os.path.exists(elec_file) or not os.path.exists(anode_file):
        print("❌ 找不到输入文件！")
        return
    bulk_elyte = Structure.from_file(elec_file)
    bulk_anode = Structure.from_file(anode_file)

    # --- 切片 ---
    print("🔪 生成基础切片...")
    # min_slab_size 决定了那一层的厚度，可以适当改大
    slab_elyte = SlabGenerator(bulk_elyte, hkl_elec, min_slab_size=12, min_vacuum_size=15, center_slab=True).get_slab()
    slab_anode = SlabGenerator(bulk_anode, hkl_anode, min_slab_size=12, min_vacuum_size=15, center_slab=True).get_slab()

    # --- ZSL ---
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.1, max_area=max_area, max_length_tol=0.1, max_angle_tol=max_angle_tol
    )

    # --- Builder ---
    print("🤖 初始化自动构建器...")
    builder = CoherentInterfaceBuilder(
        substrate_structure=slab_elyte,
        film_structure=slab_anode,
        film_miller=hkl_anode,      
        substrate_miller=hkl_elec,  
        zslgen=zsl                  
    )

    # --- 遍历 ---
    print("🔍 开始搜索并生成单层界面...")
    saved_count = 0
    available_terms = list(builder.terminations)

    for term in available_terms:
        term_label = f"{term[0]}_{term[1]}" if isinstance(term, tuple) else str(term)
        term_label = term_label.replace(" ", "")

        for interface in builder.get_interfaces(termination=term):
            
            max_s = get_max_strain(interface)
            num_atoms = len(interface)
            
            if max_s != -1 and max_s > max_strain: continue
            if num_atoms > 300 or num_atoms < 40: continue

            saved_count += 1
            
            # ================= [添加真空的核心步骤] =================
            final_struct = interface.copy()
            
            # 1. 获取当前晶格和坐标
            # 为了加真空，我们先只取 Cartesian 坐标，重构盒子
            cart_coords = final_struct.cart_coords
            species = final_struct.species
            old_lattice = final_struct.lattice
            
            # 2. 计算当前原本的厚度 (Z方向)
            z_coords = cart_coords[:, 2]
            thickness = np.max(z_coords) - np.min(z_coords)
            
            # 3. 定义新的 C 轴长度 = 实体厚度 + 真空层
            new_c_length = thickness + VACUUM_SIZE
            
            # 4. 构建新晶格 (保持 a, b, alpha, beta, gamma 不变，只改 c)
            new_lattice = Lattice.from_parameters(
                a=old_lattice.a,
                b=old_lattice.b,
                c=new_c_length, # 这里拉长了 C 轴
                alpha=old_lattice.alpha,
                beta=old_lattice.beta,
                gamma=old_lattice.gamma
            )
            
            # 5. 创建新结构
            # 注意：这时候所有原子的 Z 坐标还是老的，都在盒子底部
            final_struct = Structure(
                new_lattice,
                species,
                cart_coords,
                coords_are_cartesian=True
            )
            
            # 6. 居中操作 (美观)
            # 把实体部分移动到真空盒子的正中央
            # 计算几何中心
            center_of_mass = np.mean(final_struct.cart_coords, axis=0)
            target_center = final_struct.lattice.get_cartesian_coords([0.5, 0.5, 0.5])
            shift_vector = target_center - center_of_mass
            
            final_struct.translate_sites(
                indices=range(len(final_struct)),
                vector=shift_vector,
                frac_coords=False,
                to_unit_cell=True
            )
            
            final_struct.sort()
            # ================= [结束] =================
            
            # 保存
            strain_str = f"{max_s:.3f}" if max_s != -1 else "Unknown"
            file_name = f"SingleLayer_S{strain_str}_N{num_atoms}_{term_label}_{saved_count}.vasp"
            full_path = os.path.join(output_dir, file_name)
            
            final_struct.to(filename=full_path, fmt="poscar")
            
            print(f"   ✅ [保存] 原子数:{num_atoms} | 真空:{VACUUM_SIZE}Å | 文件:{file_name}")

            if saved_count >= 5:
                break
        
        if saved_count >= 5:
            print("🛑 已生成 5 个优质单层结构。")
            break

    if saved_count == 0:
        print(f"\n⚠️ 未找到结构。")
    else:
        print(f"\n🎉 任务完成！结构在: {output_dir}")
        print("💡 提示：用 Ovito 打开时，这应该是一个被真空包围的孤立 slab。")

if __name__ == "__main__":
    run_builder_with_vacuum()

📁 创建输出目录: Na_Na3SbS4_Data/Interface_structures_Vacuum
🔪 生成基础切片...
🤖 初始化自动构建器...
🔍 开始搜索并生成单层界面...
   ✅ [保存] 原子数:73 | 真空:20.0Å | 文件:SingleLayer_SUnknown_N73_Na_Cmmm_1_S_Cmmm_1_1.vasp
   ✅ [保存] 原子数:73 | 真空:20.0Å | 文件:SingleLayer_SUnknown_N73_Na_Cmmm_1_S_Cmmm_1_2.vasp
   ✅ [保存] 原子数:78 | 真空:20.0Å | 文件:SingleLayer_SUnknown_N78_Na_Cmmm_1_S_Cmmm_1_3.vasp
   ✅ [保存] 原子数:78 | 真空:20.0Å | 文件:SingleLayer_SUnknown_N78_Na_Cmmm_1_S_Cmmm_1_4.vasp
   ✅ [保存] 原子数:112 | 真空:20.0Å | 文件:SingleLayer_SUnknown_N112_Na_Cmmm_1_S_Cmmm_1_5.vasp
🛑 已生成 5 个优质单层结构。

🎉 任务完成！结构在: Na_Na3SbS4_Data/Interface_structures_Vacuum
💡 提示：用 Ovito 打开时，这应该是一个被真空包围的孤立 slab。


In [19]:
import os
import numpy as np
from pymatgen.core import Structure, Lattice
from pymatgen.core.surface import SlabGenerator
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

# ================= 1. 配置区域 =================
elec_file = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
anode_file = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"
output_dir = "Na_Na3SbS4_Data/Interface_Fixed_View3"

hkl_elec = (0, 0, 1)  
hkl_anode = (0, 0, 1) 

max_area = 400
max_strain = 0.05     
VACUUM_SIZE = 7.0  # 最终想要的真空层厚度

# ================= 2. 辅助函数 =================
def get_max_strain(interface):
    # (保持之前的辅助函数不变)
    strain_matrix = None
    if hasattr(interface, "film_strain"):
        strain_matrix = interface.film_strain
    elif hasattr(interface, "metadata"):
        meta = interface.metadata
        if "film_strain" in meta: strain_matrix = meta["film_strain"]
        elif "strain" in meta: strain_matrix = meta["strain"]
    if strain_matrix is not None:
        return np.max(np.abs(strain_matrix))
    return -1.0

# ================= 3. 主逻辑 =================

def run_fix_view_builder():
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    if not os.path.exists(elec_file) or not os.path.exists(anode_file):
        print("❌ 找不到输入文件！")
        return
    bulk_elyte = Structure.from_file(elec_file)
    bulk_anode = Structure.from_file(anode_file)

    # --- [关键修改 1] 切片时不加真空 ---
    print("🔪 生成紧密切片 (无真空)...")
    # min_vacuum_size=0.0 确保原子紧密堆积，不会自带缝隙
    # center_slab=False 让原子从 z=0 开始堆，方便后续拼接
    slab_elyte = SlabGenerator(bulk_elyte, hkl_elec, min_slab_size=12, min_vacuum_size=0.0, center_slab=False).get_slab()
    slab_anode = SlabGenerator(bulk_anode, hkl_anode, min_slab_size=12, min_vacuum_size=0.0, center_slab=False).get_slab()

    # --- Builder ---
    zsl = ZSLGenerator(max_area_ratio_tol=0.1, max_area=max_area, max_length_tol=0.1, max_angle_tol=0.1)
    
    print("🤖 初始化构建器...")
    builder = CoherentInterfaceBuilder(
        substrate_structure=slab_elyte,
        film_structure=slab_anode,
        film_miller=hkl_anode,      
        substrate_miller=hkl_elec,  
        zslgen=zsl                  
    )

    print("🔍 开始生成并修复视图...")
    saved_count = 0
    available_terms = list(builder.terminations)

    for term in available_terms:
        term_label = f"{term[0]}_{term[1]}" if isinstance(term, tuple) else str(term)
        term_label = term_label.replace(" ", "")

        for interface in builder.get_interfaces(termination=term):
            
            max_s = get_max_strain(interface)
            num_atoms = len(interface)
            
            if max_s != -1 and max_s > max_strain: continue
            if num_atoms > 300 or num_atoms < 40: continue

            saved_count += 1
            
            # ================= [核心修复逻辑] =================
            # 1. 获取纯净的 Cartesian 坐标
            # interface 可能包含 builder 默认加的一些真空，我们要清除它，重新定义
            struct = interface.copy()
            cart_coords = struct.cart_coords
            species = struct.species
            
            # 2. 计算原子的真实厚度 (Z方向跨度)
            z_coords = cart_coords[:, 2]
            min_z = np.min(z_coords)
            max_z = np.max(z_coords)
            real_thickness = max_z - min_z
            
            # 3. 将所有原子推到底部 (Z=0附近)，消除原有的偏移
            cart_coords[:, 2] -= min_z 
            
            # 4. 定义新的晶胞高度
            new_c = real_thickness + VACUUM_SIZE
            
            # 5. 重建结构 (使用新的晶胞高度)
            new_lattice_params = list(struct.lattice.parameters)
            new_lattice_params[2] = new_c # 修改 c 轴长度
            new_lattice = Lattice.from_parameters(*new_lattice_params)
            
            fixed_struct = Structure(
                new_lattice,
                species,
                cart_coords,
                coords_are_cartesian=True
            )
            
            # 6. [最重要的一步] 完美居中
            # 计算几何中心
            center_of_mass = np.mean(fixed_struct.cart_coords, axis=0)
            # 目标是晶胞中心
            target_center = fixed_struct.lattice.get_cartesian_coords([0.5, 0.5, 0.5])
            shift_vector = target_center - center_of_mass
            
            # 平移所有原子，并开启 to_unit_cell=True
            # 这会把挂在边界上的原子“折叠”回来，解决你图片里的断裂问题
            fixed_struct.translate_sites(
                indices=range(len(fixed_struct)),
                vector=shift_vector,
                frac_coords=False,
                to_unit_cell=True 
            )
            
            fixed_struct.sort()
            # ================= [结束] =================
            
            strain_str = f"{max_s:.3f}" if max_s != -1 else "Unknown"
            file_name = f"FixedView_S{strain_str}_N{num_atoms}_{term_label}_{saved_count}.vasp"
            full_path = os.path.join(output_dir, file_name)
            
            fixed_struct.to(filename=full_path, fmt="poscar")
            
            print(f"   ✅ [保存] 原子数:{num_atoms} | 视图已修正 | 文件:{file_name}")

            if saved_count >= 5:
                break
        
        if saved_count >= 5:
            print("🛑 已生成 5 个结构。")
            break

    if saved_count == 0:
        print(f"\n⚠️ 未找到结构。")
    else:
        print(f"\n🎉 完成！结构在: {output_dir}")
        print("💡 提示：在 Ovito 中打开，应该是一个悬浮在中间的完整单层结构。")

if __name__ == "__main__":
    run_fix_view_builder()

🔪 生成紧密切片 (无真空)...
🤖 初始化构建器...
🔍 开始生成并修复视图...
   ✅ [保存] 原子数:42 | 视图已修正 | 文件:FixedView_SUnknown_N42_Na_Cmmm_1_NaSbS2_Amm2_4_1.vasp
   ✅ [保存] 原子数:42 | 视图已修正 | 文件:FixedView_SUnknown_N42_Na_Cmmm_1_NaSbS2_Amm2_4_2.vasp
   ✅ [保存] 原子数:42 | 视图已修正 | 文件:FixedView_SUnknown_N42_Na_Cmmm_1_NaSbS2_Amm2_4_3.vasp
   ✅ [保存] 原子数:42 | 视图已修正 | 文件:FixedView_SUnknown_N42_Na_Cmmm_1_NaSbS2_Amm2_4_4.vasp
   ✅ [保存] 原子数:43 | 视图已修正 | 文件:FixedView_SUnknown_N43_Na_Cmmm_1_NaSbS2_Amm2_4_5.vasp
🛑 已生成 5 个结构。

🎉 完成！结构在: Na_Na3SbS4_Data/Interface_Fixed_View3
💡 提示：在 Ovito 中打开，应该是一个悬浮在中间的完整单层结构。


In [1]:
!ls

0a8901819275397a17cbeceae0ee7ca914e64552_job_id   Na_Na3SbS4_Data
0a8901819275397a17cbeceae0ee7ca914e64552.sub	  new-vasp-interface.ipynb
0a8901819275397a17cbeceae0ee7ca914e64552.sub.run  scripts
b53d8cb1575b4627a9d6c67528a3b3feb7f6c6f0.json	  test-interface.ipynb
dpdispatcher.log				  vasp-interface.ipynb
LiGa						  vasp-Na3SbS4.ipynb
main-3.ipynb


In [12]:
import os
import numpy as np
from pymatgen.core import Structure
from pymatgen.core.surface import SlabGenerator
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

# ================= 1. 配置区域 =================
# 输入文件路径
elec_file = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
anode_file = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"

# 界面构建参数
hkl_elec = (0, 0, 1)  
hkl_anode = (0, 0, 1) 

# ZSL 参数 (如果你发现找不到结构，试着把 max_strain 改大一点)
max_area = 400
max_strain = 0.05      # 5% 应变限制
max_angle_tol = 0.1

# 切片厚度 (影响原子总数)
# 稍微设厚一点，以便观察是否能达到 150 原子的目标
min_slab_thick = 15.0 

# ================= 2. 辅助函数 (防报错) =================
def get_max_strain(interface):
    """安全获取应变数据"""
    strain_matrix = None
    # 兼容旧版
    if hasattr(interface, "film_strain"):
        strain_matrix = interface.film_strain
    # 兼容新版
    elif hasattr(interface, "metadata"):
        meta = interface.metadata
        if "film_strain" in meta: strain_matrix = meta["film_strain"]
        elif "strain" in meta: strain_matrix = meta["strain"]
    
    if strain_matrix is not None:
        return np.max(np.abs(strain_matrix))
    return -1.0 # Unknown

# ================= 3. 主逻辑 =================

def dry_run_builder():
    print("🚀 启动界面搜索 (诊断模式 - 不生成文件)...")
    print("-" * 50)

    # --- 读取 ---
    if not os.path.exists(elec_file) or not os.path.exists(anode_file):
        print(f"❌ 错误: 找不到输入文件！")
        return

    bulk_elyte = Structure.from_file(elec_file).get_primitive_structure()
    bulk_anode = Structure.from_file(anode_file).get_primitive_structure()
    print(f"📖 输入结构已读取。")

    # --- 切片 ---
    print(f"🔪 正在切片 (厚度设定: {min_slab_thick} Å)...")
    slab_elyte = SlabGenerator(bulk_elyte, hkl_elec, min_slab_size=min_slab_thick, min_vacuum_size=15, center_slab=True).get_slab()
    slab_anode = SlabGenerator(bulk_anode, hkl_anode, min_slab_size=min_slab_thick, min_vacuum_size=15, center_slab=True).get_slab()
    
    print(f"   -> 电解质 Slab 原子数: {len(slab_elyte)}")
    print(f"   -> 钠金属 Slab 原子数: {len(slab_anode)}")

    # --- ZSL ---
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.2,
        max_area=max_area,
        max_length_tol=400,
        max_angle_tol=0.1
    )

    # --- Builder ---
    print("🤖 正在计算晶格匹配 (CoherentInterfaceBuilder)...")
    try:
        builder = CoherentInterfaceBuilder(
            substrate_structure=slab_elyte,
            film_structure=slab_anode,
            film_miller=hkl_anode,      
            substrate_miller=hkl_elec,  
            zslgen=zsl                  
        )
    except Exception as e:
        print(f"❌ 初始化 Builder 失败: {e}")
        return

    # --- 遍历预览 ---
    print("\n🔍 搜索结果预览:")
    print(f"{'ID':<5} | {'原子数':<8} ")
    print("-" * 65)

    count = 0
    available_terms = list(builder.terminations)
    
    if not available_terms:
        print("⚠️  警告: 没有找到任何匹配的界面！")
        print("   原因可能是: max_strain 设置太小，或者 max_area 太小。")
        print("   建议: 将 max_strain 增加到 0.08 或 0.1 试试。")
        return

    for term in available_terms:
        term_label = f"{term[0]}_{term[1]}" if isinstance(term, tuple) else str(term)
        term_label = term_label.replace(" ", "")

        # 遍历该终端下的所有界面
        for interface in builder.get_interfaces(termination=term):
            
            max_s = get_max_strain(interface)
            num_atoms = len(interface)
            strain_str = f"{max_s:.2%}" if max_s != -1 else "Unknown"

            # 打印信息 (不论好坏，都打印出来让你看)
            print(f"{count:<5} | {num_atoms:<8}" )
            
            count += 1
            
            # 只显示前 20 个结果，避免刷屏
            if count >= 20:
                print("...")
                print(f"🛑 预览已截断 (显示前 20 个)。")
                return

    print("-" * 65)
    print(f"💡 总共找到 {count} 个潜在结构。")
    if count > 0:
        print("✅ 看起来可以正常工作！可以去运行生成脚本了。")

if __name__ == "__main__":
    dry_run_builder()

🚀 启动界面搜索 (诊断模式 - 不生成文件)...
--------------------------------------------------
📖 输入结构已读取。
🔪 正在切片 (厚度设定: 15.0 Å)...
   -> 电解质 Slab 原子数: 24
   -> 钠金属 Slab 原子数: 6
🤖 正在计算晶格匹配 (CoherentInterfaceBuilder)...

🔍 搜索结果预览:
ID    | 原子数      
-----------------------------------------------------------------
0     | 42      
1     | 42      
2     | 48      
3     | 54      
4     | 54      
5     | 54      
6     | 60      
7     | 60      
8     | 60      
9     | 60      
10    | 72      
11    | 72      
12    | 72      
13    | 72      
14    | 72      
15    | 72      
16    | 72      
17    | 72      
18    | 72      
19    | 72      
...
🛑 预览已截断 (显示前 20 个)。


In [13]:
import os
import numpy as np
from pymatgen.core import Structure
from pymatgen.core.surface import SlabGenerator
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

# ================= 1. 配置区域 =================
elec_file = "Na_Na3SbS4_Data/structures/Na3SbS4_mp-10167.vasp" 
anode_file = "Na_Na3SbS4_Data/structures/Na_mp-127.vasp"
output_dir = "Na_Na3SbS4_Data/Interface_structures_150atoms" # 新文件夹

hkl_elec = (0, 0, 1)  
hkl_anode = (0, 0, 1) 

# ZSL 参数
max_area = 500        # [修改] 略微增大允许的面积，为了容纳更多原子
max_strain = 0.08     # [修改] 放宽到 8%，为了更容易找到大结构
max_angle_tol = 0.1

# 目标原子数设置
TARGET_ATOMS = 150
TOLERANCE = 40        # 接受范围: 110 - 190

# ================= 2. 辅助函数 =================
def get_max_strain(interface):
    strain_matrix = None
    if hasattr(interface, "film_strain"):
        strain_matrix = interface.film_strain
    elif hasattr(interface, "metadata"):
        meta = interface.metadata
        if "film_strain" in meta: strain_matrix = meta["film_strain"]
        elif "strain" in meta: strain_matrix = meta["strain"]
    if strain_matrix is not None:
        return np.max(np.abs(strain_matrix))
    return -1.0

# ================= 3. 主逻辑 =================

def run_target_size_builder():
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"📁 创建输出目录: {output_dir}")

    # --- 读取 ---
    if not os.path.exists(elec_file) or not os.path.exists(anode_file):
        print(f"❌ 找不到输入文件！")
        return
    bulk_elyte = Structure.from_file(elec_file)
    bulk_anode = Structure.from_file(anode_file)

    # --- [关键修改 1] 增加切片厚度 ---
    # 之前是 9A，现在增加到 18A (电解质) 和 20A (Na)
    # 这样可以直接让原子数翻倍
    print("🔪 生成加厚切片 (Slabs)...")
    slab_elyte = SlabGenerator(bulk_elyte, hkl_elec, min_slab_size=18, min_vacuum_size=15, center_slab=True).get_slab()
    slab_anode = SlabGenerator(bulk_anode, hkl_anode, min_slab_size=20, min_vacuum_size=15, center_slab=True).get_slab()

    # --- ZSL ---
    zsl = ZSLGenerator(
        max_area_ratio_tol=0.2,
        max_area=max_area,
        max_length_tol=0.2,
        max_angle_tol=max_angle_tol
    )

    # --- Builder ---
    print("🤖 初始化构建器...")
    builder = CoherentInterfaceBuilder(
        substrate_structure=slab_elyte,
        film_structure=slab_anode,
        film_miller=hkl_anode,      
        substrate_miller=hkl_elec,  
        zslgen=zsl                  
    )

    # --- 遍历 ---
    print(f"🔍 开始搜索 (目标原子数: {TARGET_ATOMS} ± {TOLERANCE})...")
    saved_count = 0
    available_terms = list(builder.terminations)

    for term in available_terms:
        term_label = f"{term[0]}_{term[1]}" if isinstance(term, tuple) else str(term)
        term_label = term_label.replace(" ", "")

        for interface in builder.get_interfaces(termination=term):
            
            max_s = get_max_strain(interface)
            num_atoms = len(interface)
            
            # 基础筛选
            if max_s != -1 and max_s > max_strain: continue
            
            # --- [关键修改 2] 智能扩胞 (Supercell) ---
            # 如果生成的结构太小 (例如 < 100)，我们在平面上扩展它
            final_struct = interface.copy()
            expanded_flag = False
            
            if num_atoms < (TARGET_ATOMS - TOLERANCE): # 小于 110
                # 计算需要扩大的倍数
                scale_factor = round(TARGET_ATOMS / num_atoms)
                if scale_factor >= 2:
                    # 在 a 方向或 b 方向扩胞
                    # 这里简单粗暴：如果 a < b 扩 a，否则扩 b，保持盒子尽量方
                    a_len = final_struct.lattice.a
                    b_len = final_struct.lattice.b
                    
                    if scale_factor == 2:
                        matrix = [2, 1, 1] if a_len < b_len else [1, 2, 1]
                    else: # 如果太小，统一扩 2x2
                         matrix = [2, 2, 1]
                    
                    final_struct.make_supercell(matrix)
                    num_atoms = len(final_struct) # 更新原子数
                    expanded_flag = True

            # --- 最终筛选 ---
            # 只有在目标范围内的才保存
            if abs(num_atoms - TARGET_ATOMS) > TOLERANCE:
                # 如果扩胞后还是太大或太小，就跳过
                continue

            saved_count += 1
            
            # --- 保存 ---
            strain_str = f"{max_s:.3f}" if max_s != -1 else "Unknown"
            expand_tag = "_Supercell" if expanded_flag else ""
            
            file_name = f"Target150_S{strain_str}_N{num_atoms}_{term_label}{expand_tag}_{saved_count}.vasp"
            full_path = os.path.join(output_dir, file_name)
            
            final_struct.to(filename=full_path, fmt="poscar")
            
            print(f"   ✅ [保存] 原子数:{num_atoms} | 应变:{strain_str} | 扩胞:{expanded_flag} | 终端:{term_label}")

            if saved_count >= 5:
                break
        
        if saved_count >= 5:
            print("🛑 已生成 5 个满足大小要求的结构。")
            break

    if saved_count == 0:
        print(f"\n⚠️ 未找到合适大小的结构。")
        print("建议: 检查 min_slab_size 是否设置得不够大，或者放宽 TOLERANCE。")
    else:
        print(f"\n🎉 任务完成！结果在: {output_dir}")

if __name__ == "__main__":
    run_target_size_builder()

📁 创建输出目录: Na_Na3SbS4_Data/Interface_structures_150atoms
🔪 生成加厚切片 (Slabs)...
🤖 初始化构建器...
🔍 开始搜索 (目标原子数: 150 ± 40)...
   ✅ [保存] 原子数:120 | 应变:Unknown | 扩胞:True | 终端:Na_Cmmm_1_S_Cmmm_1
   ✅ [保存] 原子数:113 | 应变:Unknown | 扩胞:False | 终端:Na_Cmmm_1_S_Cmmm_1
   ✅ [保存] 原子数:113 | 应变:Unknown | 扩胞:False | 终端:Na_Cmmm_1_S_Cmmm_1
   ✅ [保存] 原子数:113 | 应变:Unknown | 扩胞:False | 终端:Na_Cmmm_1_S_Cmmm_1
   ✅ [保存] 原子数:113 | 应变:Unknown | 扩胞:False | 终端:Na_Cmmm_1_S_Cmmm_1
🛑 已生成 5 个满足大小要求的结构。

🎉 任务完成！结果在: Na_Na3SbS4_Data/Interface_structures_150atoms
